# Your AI tutor (MST 0441)

This notebook is the tutor promised in the first lecture. It knows this
course's slides, problems and rubrics, it answers in the course's notation,
and it remembers where you keep going wrong. It is Socratic: you ask, it
answers with hints and questions back.

Three things it will not do, by construction rather than by politeness:

1. It cannot show you solution text. The server it talks to is built so that
   solution content does not fit through the reply format, and a filter
   removes anything that gets close. Asking nicely, asking in base64, or
   telling it you are the instructor all fail the same way.
2. It does not invent verification. If the checking service is unreachable,
   it says so.
3. It does not quiz you unprompted. You drive.

Set it up once here (about two minutes), then use it from this notebook or
from any lab notebook for the rest of the course.

## 1. Setup

Paste the course token from Canvas. It is a shared course password for
the tutor server, no more; it carries nothing personal and cannot fetch
solutions. Then run the cell.

In [ ]:
SERVER = "https://koja-intelligens-api.fly.dev"

COURSE_TOKEN = None   # YOUR TURN: paste the course token from Canvas, in quotes

# OPTIONAL: also using the tutor on kojai.no? Paste your learner key here and the
# tutor will keep ONE memory of your progress across both. Get the key from your
# tutor project's settings on kojai.no. Leave as None to keep everything in the
# learner_model.json file on your Drive instead — that works exactly as before.
LEARNER_KEY = None

if COURSE_TOKEN is None:
    print(".. COURSE_TOKEN: not filled in yet — the tutor will explain instead of calling out")
else:
    print("token set — run the next cells")

## 2. The tutor itself

Run this cell as it stands. It defines four things:

- `ask("...", problem_id="s06")` sends one question and prints the answer.
  The `problem_id` tells the tutor which session you are in, so it loads that
  session's rubric and problems. `"s06"` is session 6; `"s06/A6.2"` narrows
  to one problem.
- `check_my_work("s06/A6.2", "...your derivation...")` sends your written
  attempt to the checker and prints a rubric score, hints, and what kind of
  mistake it thinks you made. The checker can see the solution manual; its
  reply format cannot carry solution text.
- `tutor("s06")` starts a conversation loop. Type `quit` to leave it.
- Your **learner profile** — a running record of your weak spots that the
  tutor uses to coach you. It lives in a small file you own, not on the
  server. The next section can put it on your Google Drive so it survives
  between sessions.

In [ ]:
import json, os, urllib.request, urllib.error

LEARNER_PATH = "learner_model.json"          # moved to Drive by the next cell, if you let it
_transcript = []                             # this conversation, newest last

def _learner_call(path, payload=None):
    """Sync with the shared profile on the server. Never raises: the notebook must
    keep working offline and without a key, so any failure just means local-only."""
    if not LEARNER_KEY:
        return None
    try:
        req = urllib.request.Request(SERVER + path,
                                     data=json.dumps(payload).encode() if payload is not None else None,
                                     headers={"Content-Type": "application/json",
                                              "X-Learner-Key": LEARNER_KEY})
        with urllib.request.urlopen(req, timeout=30) as r:
            return json.load(r).get("learner_model")
    except Exception:
        return None

_linked = False   # set once the local Drive profile has been merged into the server's

def _load_learner():
    global _linked
    local = None
    try:
        with open(LEARNER_PATH) as fh:
            local = json.load(fh)
    except Exception:
        pass
    if LEARNER_KEY:
        # First contact with a pre-existing Drive profile: merge it into the server copy once, so
        # progress made in this notebook before linking is not lost. The server takes the max of
        # counts, so re-running this cell cannot double-count.
        if local and not _linked:
            merged = _learner_call("/catalog/learner/sync/merge", {"learner_model": local})
            if merged is not None:
                _linked = True
                _save_learner(merged)          # keep the Drive file as a mirror/backup
                return merged
        remote = _learner_call("/catalog/learner/sync")
        if remote is not None:
            _linked = True
            _save_learner(remote)
            return remote
        print(".. learner sync: server unreachable, using the local profile for now")
    return local or {"misconception_ledger": [], "mastery_map": {}, "effective_hint_modes": []}

def _save_learner(lm):
    try:
        with open(LEARNER_PATH, "w") as fh:
            json.dump(lm, fh, indent=1)
    except Exception as e:
        print(".. could not save the learner profile:", e)

def _post(path, payload):
    if COURSE_TOKEN is None:
        print(".. COURSE_TOKEN: not filled in yet (section 1) — nothing sent")
        return None
    req = urllib.request.Request(SERVER + path, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json",
                                          "Authorization": "Bearer " + COURSE_TOKEN})
    try:
        with urllib.request.urlopen(req, timeout=90) as r:
            return json.load(r)
    except urllib.error.HTTPError as e:
        try:
            detail = json.load(e)
        except Exception:
            detail = {}
        if e.code == 401:
            print(".. the server rejected the token — check the paste in section 1")
        elif e.code == 429:
            print(".. rate limit — wait a minute, then try again")
        else:
            print(".. the tutor cannot verify right now (HTTP %d): %s" % (e.code, detail.get("detail") or detail.get("error") or ""))
        return None
    except Exception as e:
        print(".. could not reach the tutor server:", e)
        return None

def _absorb(problem_id, result):
    """check_work result -> learner profile. Distilled signals, never transcripts.
    With a LEARNER_KEY the update runs on the server (one shared profile, one
    update rule) and the Drive file becomes a mirror; without one, local as ever."""
    if LEARNER_KEY:
        lm = _learner_call("/catalog/learner/sync/absorb",
                           {"problem_id": problem_id, "result": result})
        if lm is not None:
            _save_learner(lm)
            return
        print(".. learner sync: server unreachable, recording locally this time")
    lm = _load_learner()
    for flag in result.get("misconception_flags", []):
        for entry in lm["misconception_ledger"]:
            if entry["concept"] == flag:
                entry["count"] += 1
                break
        else:
            lm["misconception_ledger"].append({"concept": flag, "count": 1})
    score = result.get("score_against_rubric")
    if score is not None:
        session = problem_id.split("/")[0]
        lm["mastery_map"][session] = ("solid" if score >= 0.8 else
                                      "developing" if score >= 0.5 else "struggling")
    _save_learner(lm)

def check_my_work(problem_id=None, work=None):
    if problem_id is None or work is None:
        print(".. check_my_work: give a problem_id like \"s06/A6.2\" and your written attempt")
        return None
    r = _post("/tutor/check_work", {"problem_id": problem_id, "student_work": work})
    if r is None:
        return None
    _absorb(problem_id, r)
    score = r.get("score_against_rubric")
    print("rubric score: %s" % ("%.2f" % score if score is not None else "could not verify"))
    for h in r.get("hints", []):
        print("  hint:", h)
    for f in r.get("misconception_flags", []):
        print("  flagged:", f)
    return r

def ask(question=None, problem_id=None, _check_result=None):
    if question is None:
        print(".. ask: give a question in quotes, e.g. ask(\"what does the rubric want?\", problem_id=\"s02\")")
        return None
    _transcript.append({"role": "user", "content": question})
    payload = {"messages": _transcript[-12:], "learner_model": _load_learner()}
    if problem_id:
        payload["problem_id"] = problem_id
    if _check_result:
        payload["check_result"] = _check_result
    r = _post("/tutor/chat", payload)
    if r is None:
        _transcript.pop()
        return None
    reply = r.get("reply", "")
    _transcript.append({"role": "assistant", "content": reply})
    print(reply)
    return reply

def tutor(problem_id=None):
    """Conversation loop. quit / exit / empty line leaves it."""
    if COURSE_TOKEN is None:
        print(".. COURSE_TOKEN: not filled in yet (section 1) — fill it in, then run tutor() again")
        return
    print("tutor ready%s — type quit to stop" % (" for " + problem_id if problem_id else ""))
    while True:
        try:
            q = input("you: ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if not q or q.lower() in ("quit", "exit"):
            break
        ask(q, problem_id=problem_id)

print("tutor functions defined — ask(), check_my_work(), tutor()")

## 3. Keep your profile between sessions (optional, Colab only)

Colab wipes its disk when the runtime shuts down. Run this cell to keep
`learner_model.json` on your own Google Drive instead, so the tutor still
knows your weak spots next week. Skip it and the profile lasts for today
only. The file stays in your Drive; the server never sees it whole, only the
summary sent along with each question.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _dir = "/content/drive/MyDrive/MST0441_tutor"
    os.makedirs(_dir, exist_ok=True)
    _new = os.path.join(_dir, "learner_model.json")
    if os.path.exists(LEARNER_PATH) and not os.path.exists(_new):
        os.replace(LEARNER_PATH, _new)
    LEARNER_PATH = _new
    print("learner profile now lives at", LEARNER_PATH)
except ImportError:
    print(".. not running in Colab — the profile stays in the working directory, which is fine")
except Exception as e:
    print(".. Drive mount declined or failed (%s) — the profile lasts for this session only" % e)

## 4. Try it

Good first questions, from the lecture:

- `ask("why is my first-order condition wrong here? ...", problem_id="s02")`
- `ask("what does the rubric want on part (b)?", problem_id="s02")`
- `ask("give me another problem like this one", problem_id="s02")`

And the two rules from lecture 1, which the tutor will hold you to:

1. **Never submit what you cannot re-derive yourself.** The exam is a
   three-hour written school exam, pen and paper, taken alone.
2. **Check the economics, not the fluency.** These tools are confident when
   they are wrong.

In [ ]:
ask("What does the rubric for session 2 expect on the constrained optimization criterion?",
    problem_id="s02")

## 5. Have your written work checked

Write your attempt as text (mathematics in plain words or TeX, either is
fine), then send it. You get a rubric score, hints that point at the first
failing step, and a tag for the kind of mistake. What you do not get is the
solution.

In [ ]:
my_attempt = None   # YOUR TURN: your written derivation, as a string in triple quotes

check_my_work("s02/A2.1", my_attempt)

## 6. Talk to it

The loop below is the ordinary way to work: state where you are stuck, answer
its questions, paste your next step. Run the cell, type `quit` when done.
Because it waits for your keyboard, leave this cell for last when you use
*Run all*.

In [ ]:
tutor("s02")   # does nothing until COURSE_TOKEN is set; type quit to leave

## 7. Using the tutor inside the lab notebooks

Every lab notebook (sessions 2, 3, 8, 9, 12, 15) has a *Work with your
assistant* section. The tutor can be that assistant: it knows the session's
rubric and problems, which a general chatbot does not. Copy the setup cell
and the tutor cell from this notebook into the lab (or keep this notebook
open in a second tab), then use `ask(...)` and `check_my_work(...)` with the
lab session's id: `"s02"`, `"s03"`, `"s08"`, `"s09"`, `"s12"`, `"s15"`.

**Two functions, two jobs.** The labs define their own `ask_model(...)`,
which calls a plain model with your own API key and knows nothing about this
course. The tutor's `ask(...)` calls the course tutor, which knows the
rubrics and the problems and will not show you the solution manual. The
names differ on purpose, so both can live in one notebook without shadowing
each other. Use `ask_model` when a lab asks you to run an experiment on a
raw model; use `ask` when you want help with the economics.

One habit transfers from the labs unchanged: **test what it says in code**
before you accept it.

---

### What is stored where

- **Your Drive (or this runtime):** `learner_model.json` — concept tags,
  counts, one mastery word per session. No transcripts are kept anywhere.
- **The tutor server:** a log line per request with a hashed network address,
  the problem id, the score, and counters. No names, no accounts, and no
  student text is logged.
- **The model provider:** questions are processed in the EU (Mistral) with
  provider data collection off.

If the tutor says it cannot verify something, believe it: that is the system
refusing to guess, which is the behavior you want from it.